<u><h1>CPSC 440 Project: Road Traffic Dataset Analysis</h1></u>

Reproducing: "Evaluating Road Crash Severity Prediction with Balanced Ensemble Models"
By Alexei Roudnitski (2024) — Published in Findings
DOI: https://doi.org/10.32866/001c.116820

Data source: https://opendata.transport.nsw.gov.au/dataset/nsw-crash-data

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    VotingClassifier,
)
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [4]:
# load data
DATA_PATH = "nsw_road_crash_data_2018-2022_crash.csv"

df = pd.read_csv(DATA_PATH, encoding="latin1")

In [5]:
# select columns for data analysis
TARGET = "Degree of crash - detailed"

FEATURES = [
    "Day of week of crash",
    "Distance",
    "Identifying feature type",
    "Type of location",
    "Urbanisation",
    "Alignment",
    "Street lighting",
    "Road surface",
    "Surface condition",
    "Weather",
    "Natural lighting",
    "Signals operation",
    "Other traffic control",
    "Speed limit",
    "Road classification (admin)",
    "First impact type",
    "Key TU type",
    "Other TU type",
    "No. of traffic units involved",
]

df = df[[TARGET] + FEATURES].copy()
print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")

Dataset shape: (95873, 20)

Target distribution:
Degree of crash - detailed
Non-casualty (towaway)    30771
Moderate Injury           26740
Serious Injury            19700
Minor/Other Injury        17220
Fatal                      1442
Name: count, dtype: int64


In [ ]:
def remove_unknown_strings_only(df):
    """
    Removes rows where any of the 20 columns contain the word 'unknown',
    but preserves rows that contain actual NaN/Null values.
    """
    # The 20 variables identified in the paper
    features = FEATURES
    
    # Create a mask for 'unknown'
    # We cast to string to use .str.contains, but we must handle NaNs 
    # so they don't get caught in the 'unknown' filter.
    contains_unknown = df[features].apply(
        lambda col: col.astype(str).str.contains('unknown', case=False, na=False)
    ).any(axis=1)
    
    # Filter the dataframe
    df_no_unknowns = df[~contains_unknown].copy()
    
    # Validation
    unknown_count = contains_unknown.sum()
    null_count = df_no_unknowns[features].isnull().any(axis=1).sum()
    
    print(f"Rows removed due to 'unknown' string: {unknown_count}")
    print(f"Rows remaining that still contain NaNs: {null_count}")
    print(f"New dataset shape: {df_no_unknowns.shape}")
    
    return df_no_unknowns

cleaned_df = remove_unknown_strings_only(df)
print(f"\nTarget distribution:\n{cleaned_df[TARGET].value_counts()}")

Rows removed due to 'unknown' string: 39076
Rows remaining that still contain NaNs: 20228
New dataset shape: (56797, 20)

Target distribution:
Degree of crash - detailed
Moderate Injury           20224
Serious Injury            17979
Non-casualty (towaway)     9657
Minor/Other Injury         7528
Fatal                      1409
Name: count, dtype: int64


In [ ]:
# import pandas as pd
# import prince
# from pgmpy.estimators import HillClimbSearch
# from pgmpy.structure_score import BIC
# from pgmpy.models import BayesianNetwork
# from pgmpy.inference import VariableElimination
# from sklearn.metrics import accuracy_score

# # 1. Load Data (Using a smaller subset of columns to keep the graphical model tractable)
# # df = pd.read_csv("nsw_road_crash_data_2018-2022_crash.xlsx - Crash Data Table.csv")

# gm_features = ['Day of week of crash', 
#                'Speed limit', 
#                'Weather', 
#                'Key TU type', 
#                'No. of traffic units involved',
#                'Distance',
#                'Degree of crash - detailed']

# data_gm = df[gm_features].dropna().copy()

# #print(data_gm)
# data_gm['No. of traffic units involved'] = data_gm['No. of traffic units involved'].astype(float)
# print(data_gm['No. of traffic units involved'].dtype)


float64


In [ ]:

# # ==========================================
# # Method A: Factor Analysis of Mixed Data (FAMD)
# # ==========================================
# print("Running Factor Analysis of Mixed Data (FAMD)...")
# # FAMD handles both numeric and categorical data natively
# famd = prince.FAMD(
#     n_components=3,
#     n_iter=10,
#     random_state=42
# )

# # Exclude target for the dimensionality reduction step
# X_famd = data_gm.drop('Degree of crash - detailed', axis=1)
# famd_transformed = famd.fit(X_famd)

# print("Explained Variance by Component:")
# print(famd.percentage_of_variance_)


Running Factor Analysis of Mixed Data (FAMD)...


MemoryError: Unable to allocate 776. MiB for an array with shape (95873, 1061) and data type float64

In [ ]:

# ==========================================
# Method B: Probabilistic Graphical Model (Bayesian Network)
# ==========================================
print("\nRunning Graphical Model (Structure Learning)...")

# Discretize continuous variables if necessary (though our selected features are mostly discrete)
# Use a sample of the data to speed up structure learning algorithms
sample_data = data_gm.sample(n=3000, random_state=42)

# Structure Learning using Hill Climb Search and BIC Score
hc = HillClimbSearch(sample_data)
best_model_structure = hc.estimate(scoring_method=BIC(sample_data))

print("Learned Edges in the Bayesian Network:")
print(best_model_structure.edges())

# Build the Bayesian Network
bn_model = BayesianNetwork(best_model_structure.edges())
bn_model.fit(sample_data)

# Perform Inference (Example: Predicting severity based on a Saturday crash in a 100km/h zone)
inference = VariableElimination(bn_model)
evidence = {'Day of week of crash': 'Saturday', 'Speed limit': '100 km/h'}

# Check if 'Degree of crash' is in the learned network before querying
if 'Degree of crash' in bn_model.nodes():
    query_result = inference.query(variables=['Degree of crash - detailed'], evidence=evidence)
    print("\nInference Result for Saturday in 100km/h zone:")
    print(query_result)
else:
    print("\nThe structure learning did not directly connect the chosen evidence to 'Degree of crash'.")

In [ ]:
# start here :)